In [28]:
from datetime import datetime

if not os.path.exists('vhi_data'):
    os.makedirs('vhi_data')

def download_vhi_data():
    for province_id in range(1, 28): 
        existing_files = [f for f in os.listdir('vhi_data') if f.startswith(f"vhi_id_{province_id}_")]
        
        if existing_files:
            print(f"Область {province_id} вже загружена: {existing_files[0]}. Пропуск.")
            continue
            
        now = datetime.now().strftime("%Y%m%d_%H%M%S")
        filename = f"vhi_id_{province_id}_{now}.csv"
        url = f"https://www.star.nesdis.noaa.gov/smcd/emb/vci/VH/get_TS_admin.php?country=UKR&provinceID={province_id}&year1=1981&year2=2024&type=Mean"
        
        print(f"Загрузка данних для області {province_id}...")
        try:
            with urllib.request.urlopen(url) as response:
                content = response.read().decode('utf-8')
                
                clean_content = content.replace("<pre>", "").replace("</pre>", "").replace("<br>", "")
                
                with open(os.path.join('vhi_data', filename), 'w') as f:
                    f.write(clean_content)
            print(f"Файл {filename} успішно збережено.")
        except Exception as e:
            print(f"Не вдалось загрузити область {province_id}: {e}")

download_vhi_data()

Область 1 вже загружена: vhi_id_1_20260420_013933.csv. Пропуск.
Область 2 вже загружена: vhi_id_2_20260420_013939.csv. Пропуск.
Область 3 вже загружена: vhi_id_3_20260420_013939.csv. Пропуск.
Область 4 вже загружена: vhi_id_4_20260420_013941.csv. Пропуск.
Область 5 вже загружена: vhi_id_5_20260420_013942.csv. Пропуск.
Область 6 вже загружена: vhi_id_6_20260420_013943.csv. Пропуск.
Область 7 вже загружена: vhi_id_7_20260420_013943.csv. Пропуск.
Область 8 вже загружена: vhi_id_8_20260420_013944.csv. Пропуск.
Область 9 вже загружена: vhi_id_9_20260420_013945.csv. Пропуск.
Область 10 вже загружена: vhi_id_10_20260420_013946.csv. Пропуск.
Область 11 вже загружена: vhi_id_11_20260420_013947.csv. Пропуск.
Область 12 вже загружена: vhi_id_12_20260420_013948.csv. Пропуск.
Область 13 вже загружена: vhi_id_13_20260420_013949.csv. Пропуск.
Область 14 вже загружена: vhi_id_14_20260420_013949.csv. Пропуск.
Область 15 вже загружена: vhi_id_15_20260420_013950.csv. Пропуск.
Область 16 вже загружена: vh

In [29]:
import pandas as pd
import urllib.request
import os
import re
import io
import glob
from datetime import datetime

ua_regions = {
    1: {"name": "Черкаська", "new_id": 22}, 2: {"name": "Чернігівська", "new_id": 24},
    3: {"name": "Чернівецька", "new_id": 23}, 4: {"name": "Крим", "new_id": 25},
    5: {"name": "Дніпропетровська", "new_id": 3}, 6: {"name": "Донецька", "new_id": 4},
    7: {"name": "Івано-Франківська", "new_id": 8}, 8: {"name": "Харківська", "new_id": 19},
    9: {"name": "Херсонська", "new_id": 20}, 10: {"name": "Хмельницька", "new_id": 21},
    11: {"name": "Київська", "new_id": 9}, 12: {"name": "м. Київ", "new_id": 26},
    13: {"name": "Кіровоградська", "new_id": 10}, 14: {"name": "Луганська", "new_id": 11},
    15: {"name": "Львівська", "new_id": 12}, 16: {"name": "Миколаївська", "new_id": 13},
    17: {"name": "Одеська", "new_id": 14}, 18: {"name": "Полтавська", "new_id": 15},
    19: {"name": "Рівненська", "new_id": 16}, 20: {"name": "м. Севастополь", "new_id": 27},
    21: {"name": "Сумська", "new_id": 17}, 22: {"name": "Тернопільська", "new_id": 18},
    23: {"name": "Закарпатська", "new_id": 6}, 24: {"name": "Вінницька", "new_id": 1},
    25: {"name": "Волинська", "new_id": 2}, 26: {"name": "Запорізька", "new_id": 7},
    27: {"name": "Житомирська", "new_id": 5}
}

In [30]:
def download_and_load_vhi(folder='vhi_data'):
    if not os.path.exists(folder):
        os.makedirs(folder)

    for p_id in range(1, 28):
        existing = [f for f in os.listdir(folder) if f.startswith(f"vhi_id_{p_id}_")]
        if not existing:
            now = datetime.now().strftime("%Y%m%d_%H%M%S")
            url = f"https://www.star.nesdis.noaa.gov/smcd/emb/vci/VH/get_TS_admin.php?country=UKR&provinceID={p_id}&year1=1981&year2=2024&type=Mean"
            try:
                with urllib.request.urlopen(url) as response:
                    with open(os.path.join(folder, f"vhi_id_{p_id}_{now}.csv"), 'wb') as f:
                        f.write(response.read())
            except Exception as e: print(f"Error ID {p_id}: {e}")

    all_dfs = []
    for path in glob.glob(f"{folder}/*.csv"):
        noaa_id = int(os.path.basename(path).split('_')[2])
        with open(path, 'r') as f:
            content = re.sub(r'<[^>]+>', '', f.read()) # Убираем HTML [cite: 35]
        
        df = pd.read_csv(io.StringIO(content), 
                         skiprows=1, 
                         names=['Year', 'Week', 'SMN', 'SMT', 'VCI', 'TCI', 'VHI', 'Extra'], 
                         index_col=False,
                         skipinitialspace=True)
        df = df.apply(pd.to_numeric, errors='coerce').dropna(subset=['VHI'])
        
        df['Province_Name'] = ua_regions[noaa_id]['name']
        df['Province_ID'] = ua_regions[noaa_id]['new_id']
        all_dfs.append(df)

    return pd.concat(all_dfs, ignore_index=True)

vhi_df = download_and_load_vhi()
print("Данні загружені. Розмір:", vhi_df.shape)

Данні загружені. Розмір: (60372, 10)


In [31]:
def get_vhi_year(df, province_id, year):
    return df[(df['Province_ID'] == province_id) & (df['Year'] == year)][['Week', 'VHI']]

def get_vhi_range(df, provinces, year_start, year_end):
    return df[(df['Province_ID'].isin(provinces)) & (df['Year'] >= year_start) & (df['Year'] <= year_end)]

def get_stats(df, province_id, year):
    subset = df[(df['Province_ID'] == province_id) & (df['Year'] == year)]['VHI']
    return {"Max": subset.max(), "Min": subset.min(), "Mean": subset.mean()}

print("VHI для Вінницької (ID 1) за 2020 рік:")
print(get_vhi_year(vhi_df, 1, 2020).head())

VHI для Вінницької (ID 1) за 2020 рік:
       Week    VHI
35516   1.0  40.92
35517   2.0  43.19
35518   3.0  44.74
35519   4.0  45.29
35520   5.0  44.80


In [32]:
import pandas as pd
import timeit
from datetime import datetime
from sklearn.preprocessing import StandardScaler, MinMaxScaler

def load_and_prepare_data(file_path):
    df = pd.read_csv(file_path, sep=';', low_memory=False)
    df.replace('?', pd.NA, inplace=True)
    df = df.dropna()
    numeric_cols = ['Global_active_power', 'Global_reactive_power', 'Voltage', 'Global_intensity', 'Sub_metering_1', 'Sub_metering_2', 'Sub_metering_3']
    df[numeric_cols] = df[numeric_cols].apply(pd.to_numeric)
    return df, numeric_cols

df_power, cols = load_and_prepare_data('household_power_consumption.txt')
print(f"Дані завантажено. Кількість записів: {len(df_power)}")

Дані завантажено. Кількість записів: 2049280


In [33]:
def execute_tasks(df, numeric_cols):
    t1 = timeit.timeit(lambda: df[df['Global_active_power'] > 5.0], number=1)
    print(f"Завдання 5.1 (Потужність > 5 кВт): виконано за {t1:.4f} сек.")

    res2 = df[(df['Global_intensity'].between(19, 20)) & (df['Sub_metering_2'] > df['Sub_metering_3'])]
    print(f"Завдання 5.2 (Струм 19-20 А): знайдено {len(res2)} записів.")

    sample_df = df.sample(n=500000, replace=False)
    print("Завдання 5.3 (Середні значення для 500,000 випадкових записів):")
    print(sample_df[['Sub_metering_1','Sub_metering_2','Sub_metering_3']].mean())

    df['Time'] = pd.to_datetime(df['Time'], format='%H:%M:%S').dt.time
    limit_time = datetime.strptime("18:00:00", "%H:%M:%S").time()
    filtered = df[(df['Time'] > limit_time) & (df['Global_active_power'] > 6.0) &
                  (df['Sub_metering_2'] > df['Sub_metering_1']) & (df['Sub_metering_2'] > df['Sub_metering_3'])]
    mid = len(filtered) // 2
    res4 = pd.concat([filtered.iloc[:mid:3], filtered.iloc[mid::4]])
    print(f"Завдання 5.4 (Складна вибірка): отримано {len(res4)} рядків.")

    std_scaler = StandardScaler()
    df_std = std_scaler.fit_transform(df[numeric_cols])
    norm_scaler = MinMaxScaler()
    df_norm = norm_scaler.fit_transform(df[numeric_cols])

    p_corr = df['Global_active_power'].corr(df['Global_intensity'], method='pearson')
    s_corr = df['Global_active_power'].corr(df['Global_intensity'], method='spearman')
    print(f"Завдання 5.6 (Коефіцієнт Пірсона): {p_corr:.3f}, (Спірмена): {s_corr:.3f}")

    df['Month'] = pd.to_datetime(df['Date'], dayfirst=True).dt.month
    ohe = pd.get_dummies(df['Month'], prefix='Місяць').head()
    print("Завдання 5.7 (Результат One Hot Encoding):")
    display(ohe)

execute_tasks(df_power, cols)

Завдання 5.1 (Потужність > 5 кВт): виконано за 0.0033 сек.
Завдання 5.2 (Струм 19-20 А): знайдено 2509 записів.
Завдання 5.3 (Середні значення для 500,000 випадкових записів):
Sub_metering_1    1.112338
Sub_metering_2    1.300856
Sub_metering_3    6.462680
dtype: float64
Завдання 5.4 (Складна вибірка): отримано 310 рядків.
Завдання 5.6 (Коефіцієнт Пірсона): 0.999, (Спірмена): 0.995
Завдання 5.7 (Результат One Hot Encoding):


,Місяць_1,Місяць_2,Місяць_3,Місяць_4,Місяць_5,Місяць_6,Місяць_7,Місяць_8,Місяць_9,Місяць_10,Місяць_11,Місяць_12
0,False,False,False,False,False,False,False,False,False,False,False,True
1,False,False,False,False,False,False,False,False,False,False,False,True
2,False,False,False,False,False,False,False,False,False,False,False,True
3,False,False,False,False,False,False,False,False,False,False,False,True
4,False,False,False,False,False,False,False,False,False,False,False,True
